# Câu 48:
Viết thuật toán tìm giá trị kỳ dị lớn nhất của ma trận $A$. Áp dụng cho một ma trận cấp 5.

## Giá trị kỳ dị lớn nhất (Power Method trên $A^TA$)

### Bài toán

Cho ma trận $A \in \text{mat}(m \times n, \mathbb{R})$ bất kỳ. Tìm **giá trị kỳ dị lớn nhất** $\sigma_1$ — tức chuẩn phổ $\|A\|_2$.

### Cơ sở lý thuyết

Giá trị kỳ dị của $A$ là căn bậc hai của giá trị riêng của ma trận **đối xứng nửa xác định dương** $G = A^T A$ (cấp $n \times n$):

$$\sigma_i = \sqrt{\lambda_i(A^T A)}, \qquad \lambda_1 \geq \lambda_2 \geq \cdots \geq 0$$

Do đó:

$$\sigma_1 = \sqrt{\lambda_{\max}(A^T A)}$$

Áp dụng **phương pháp lũy thừa** (Câu 45) cho $G = A^TA$ để lấy $\lambda_1$ và **vector kỳ dị phải** $v_1$. Khi đó **vector kỳ dị trái**:

$$u_1 = \frac{A v_1}{\sigma_1}$$

và bộ ba thoả $A v_1 = \sigma_1 u_1$, $\;A^T u_1 = \sigma_1 v_1$.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $m \times n$, sai số $\varepsilon$

**Đầu ra:** $\sigma_1$, cùng $u_1, v_1$

**Bước 1.** Lập $G \leftarrow A^T A$ (đối xứng, nửa xác định dương).

**Bước 2.** Lũy thừa trên $G$ → $(\lambda_1, v_1)$.

**Bước 3.** $\sigma_1 \leftarrow \sqrt{\lambda_1}$, $\;u_1 \leftarrow A v_1 / \sigma_1$.

---

### Lưu ý

- **Chọn $A^TA$ hay $AA^T$:** Nếu $m > n$ (nhiều hàng), $A^TA$ cấp $n$ nhỏ hơn — nên tính trên $A^TA$. Nếu $m < n$, dùng $AA^T$ cấp $m$ cho rẻ.
- **Ổn định:** $G = A^TA$ làm **bình phương số điều kiện** ($\kappa(G) = \kappa(A)^2$), khuếch đại sai số khi $A$ gần suy biến. Với $\sigma_1$ (giá trị lớn) thì không đáng ngại; vấn đề chỉ nảy sinh với $\sigma$ nhỏ.
- **Quan hệ với chuẩn:** $\sigma_1 = \|A\|_2$ là hệ số khuếch đại lớn nhất: $\max_{x \neq 0} \|Ax\| / \|x\|$.

In [1]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


# ── Phương pháp lũy thừa (như Câu 45) ────────────────────────────
def power_method(A, eps=1e-12, max_iter=5000):
    A = np.array(A, dtype=float)
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        y = A @ x
        lam = x @ y / (x @ x)              # thương Rayleigh
        x = y / np.linalg.norm(y)
        if abs(lam - lam_old) < eps:
            return lam, x, k
        lam_old = lam
    return lam, x, max_iter


# ── Giá trị kỳ dị lớn nhất ───────────────────────────────────────
def largest_singular(A, eps=1e-12):
    """
    Trả về σ₁ (kỳ dị lớn nhất), u₁ (kỳ dị trái), v₁ (kỳ dị phải), số vòng.
    """
    A = np.array(A, dtype=float)
    G = A.T @ A                            # đối xứng nửa xác định dương
    lam, v, k = power_method(G, eps=eps)   # λmax và vector kỳ dị phải
    sigma = np.sqrt(max(lam, 0.0))
    u = A @ v / sigma                      # vector kỳ dị trái
    return sigma, u, v, k


# ====== Áp dụng với ma trận cấp 5 ======
if __name__ == "__main__":
    A = np.array([
        [ 4.,  1.,  2.,  0.,  1.],
        [ 1.,  3., -1.,  2.,  0.],
        [ 2., -1.,  5.,  1.,  1.],
        [ 0.,  2.,  1.,  4., -1.],
        [ 1.,  0.,  1., -1.,  3.],
    ])

    sigma, u, v, k = largest_singular(A)
    print(f"Giá trị kỳ dị lớn nhất  σ₁ = {sigma:.10f}   ({k} vòng lặp)")
    print("\nVector kỳ dị phải v₁:")
    print("  [" + "  ".join(f"{t:8.5f}" for t in v) + "]")
    print("Vector kỳ dị trái  u₁:")
    print("  [" + "  ".join(f"{t:8.5f}" for t in u) + "]")

    # ── Kiểm tra ──
    print("\n=== Kiểm tra ===")
    print(f"  ‖A·v₁ - σ₁·u₁‖   = {_sci(np.linalg.norm(A @ v - sigma * u))}")
    print(f"  ‖Aᵀ·u₁ - σ₁·v₁‖  = {_sci(np.linalg.norm(A.T @ u - sigma * v))}")
    sig_np = np.linalg.svd(A, compute_uv=False)[0]
    print(f"  σ₁ (numpy.svd)   = {sig_np:.10f}")
    print(f"  Sai số so numpy  = {_sci(abs(sigma - sig_np))}"
          f"  →  {'ĐÚNG ✓' if abs(sigma - sig_np) < 1e-8 else 'SAI ✗'}")
    print(f"  σ₁ = ‖A‖₂ ?      chuẩn phổ numpy = {np.linalg.norm(A, 2):.10f}")


Giá trị kỳ dị lớn nhất  σ₁ = 7.1167650777   (37 vòng lặp)

Vector kỳ dị phải v₁:
  [ 0.58081   0.04825   0.74218   0.18001   0.27764]
Vector kỳ dị trái  u₁:
  [ 0.58081   0.04825   0.74218   0.18001   0.27764]

=== Kiểm tra ===
  ‖A·v₁ - σ₁·u₁‖   = 0
  ‖Aᵀ·u₁ - σ₁·v₁‖  = 3.3789×10⁻⁷
  σ₁ (numpy.svd)   = 7.1167650777
  Sai số so numpy  = 5.5067×10⁻¹⁴  →  ĐÚNG ✓
  σ₁ = ‖A‖₂ ?      chuẩn phổ numpy = 7.1167650777


# Câu 49:
Viết thuật toán tìm khai triển kỳ dị của ma trận $A$. Áp dụng cho ma trận cấp $7 \times 4$.

## Khai triển kỳ dị (SVD) bằng Lũy thừa + Xuống thang

### Bài toán

Cho ma trận $A \in \text{mat}(m \times n, \mathbb{R})$. Tìm **khai triển kỳ dị**:

$$A = U \Sigma V^T = \sum_{i=1}^{r} \sigma_i \, u_i v_i^T$$

với $\sigma_1 \geq \sigma_2 \geq \cdots \geq \sigma_r > 0$ (các giá trị kỳ dị), $\{u_i\}$ trực chuẩn trong $\mathbb{R}^m$ (**vector kỳ dị trái**), $\{v_i\}$ trực chuẩn trong $\mathbb{R}^n$ (**vector kỳ dị phải**), $r = \text{rank}(A)$.

### Cơ sở lý thuyết

Các $\sigma_i^2$ là giá trị riêng của $G = A^T A$ (đối xứng), và $v_i$ là vector riêng tương ứng. Với mỗi cặp:

$$\sigma_i = \sqrt{\lambda_i(A^TA)}, \qquad u_i = \frac{A v_i}{\sigma_i}$$

Áp dụng **lũy thừa + xuống thang** (Câu 47) cho $G = A^TA$ (cấp $n$, chọn chiều nhỏ hơn giữa $m, n$) để bóc lần lượt $(\lambda_i, v_i)$, rồi suy ra $\sigma_i, u_i$.

---

### Thuật toán

**Đầu vào:** Ma trận $A$ cấp $m \times n$, hạng cần lấy $r$, sai số $\varepsilon$

**Đầu ra:** $\Sigma = \text{diag}(\sigma_i)$, $U = [u_1 \cdots u_r]$, $V = [v_1 \cdots v_r]$

**Bước 1.** $G \leftarrow A^T A$.

**Bước 2 — Vòng lặp** $i = 1, \ldots, r$:

&emsp;**2.1.** $(\lambda_i, v_i) \leftarrow$ **PowerMethod**$(G)$.

&emsp;**2.2.** $\sigma_i \leftarrow \sqrt{\lambda_i}$; nếu $\sigma_i \approx 0$ thì dừng (hết hạng).

&emsp;**2.3.** $u_i \leftarrow A v_i / \sigma_i$.

&emsp;**2.4.** Xuống thang: $G \leftarrow G - \lambda_i \, v_i v_i^T$.

**Bước 3 — Kiểm tra.** $A \approx U \Sigma V^T$; $\;U^T U = V^T V = I$.

---

### Lưu ý

- **Chọn $A^TA$ ($7\times4$):** Với ma trận $7 \times 4$, $A^TA$ chỉ cấp $4$ (thay vì $AA^T$ cấp $7$) — rẻ hơn và có tối đa $4$ giá trị kỳ dị khác 0.
- **SVD gọn (thin SVD):** Ở đây $U$ có cấp $m \times r$, $\Sigma$ cấp $r \times r$, $V$ cấp $n \times r$ — chỉ giữ $r$ giá trị kỳ dị khác 0, đủ để tái tạo $A$.
- **Sai số tích luỹ:** Giống Câu 47, các $\sigma$ nhỏ tính sau kém chính xác hơn do xuống thang lặp. Với ứng dụng cần độ chính xác cao, dùng thuật toán SVD chuẩn (Golub–Kahan).
- **Ứng dụng:** SVD là nền của **nén ảnh** (giữ $k$ giá trị kỳ dị lớn nhất — xấp xỉ hạng thấp), **số điều kiện** (Câu 50), và **giả nghịch đảo** Moore–Penrose.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def power_method(A, eps=1e-12, max_iter=5000):
    A = np.array(A, dtype=float)
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        y = A @ x
        lam = x @ y / (x @ x)
        x = y / np.linalg.norm(y)
        if abs(lam - lam_old) < eps:
            return lam, x, k
        lam_old = lam
    return lam, x, max_iter


# ── SVD bằng lũy thừa + xuống thang trên G = AᵀA ─────────────────
def svd_deflation(A, rank=None, eps=1e-12):
    """
    Khai triển kỳ dị A = U Σ Vᵀ.

    Trả về:
        sigmas : mảng σ_i (giảm dần)
        U      : m×r, cột = vector kỳ dị trái
        V      : n×r, cột = vector kỳ dị phải
    """
    A = np.array(A, dtype=float)
    m, n = A.shape
    r = min(m, n) if rank is None else rank
    G = A.T @ A                              # cấp n, đối xứng ≥ 0
    sigmas, U, V = [], [], []
    for _ in range(r):
        lam, v, _ = power_method(G, eps=eps)
        sigma = np.sqrt(max(lam, 0.0))
        if sigma < 1e-12:
            break                            # hết hạng
        u = A @ v / sigma
        sigmas.append(sigma); V.append(v); U.append(u)
        G = G - lam * np.outer(v, v)         # xuống thang Hotelling
    return np.array(sigmas), np.array(U).T, np.array(V).T


# ====== Áp dụng với ma trận 7×4 ======
if __name__ == "__main__":
    A = np.array([
        [ 2.,  1.,  0.,  1.],
        [ 1.,  3.,  1.,  0.],
        [ 0.,  1.,  2.,  1.],
        [ 1.,  0.,  1.,  2.],
        [ 2.,  1.,  1.,  0.],
        [ 0.,  2.,  1.,  1.],
        [ 1.,  1.,  0.,  2.],
    ])
    m, n = A.shape

    sigmas, U, V = svd_deflation(A)
    print(f"Ma trận A cấp {m}×{n},  số giá trị kỳ dị = {len(sigmas)}\n")
    print("Giá trị kỳ dị:", "  ".join(f"σ{i+1} = {s:.6f}" for i, s in enumerate(sigmas)))

    print("\nU (vector kỳ dị trái, mỗi cột 1 vector):")
    for row in U:
        print("  " + "  ".join(f"{t:8.4f}" for t in row))
    print("\nV (vector kỳ dị phải, mỗi cột 1 vector):")
    for row in V:
        print("  " + "  ".join(f"{t:8.4f}" for t in row))

    # ── Kiểm tra ──
    A_rec = U @ np.diag(sigmas) @ V.T
    print("\n=== Kiểm tra ===")
    print(f"  ‖A - UΣVᵀ‖   = {_sci(np.max(np.abs(A - A_rec)))}   (tái tạo A)")
    print(f"  |UᵀU - I|    = {_sci(np.max(np.abs(U.T @ U - np.eye(len(sigmas)))))}")
    print(f"  |VᵀV - I|    = {_sci(np.max(np.abs(V.T @ V - np.eye(len(sigmas)))))}")
    sig_np = np.linalg.svd(A, compute_uv=False)
    print(f"  σ (numpy)    :", "  ".join(f"{s:.6f}" for s in sig_np))
    print(f"  Sai số σ     = {_sci(np.max(np.abs(sigmas - sig_np)))}"
          f"  →  {'ĐÚNG ✓' if np.max(np.abs(sigmas - sig_np)) < 1e-6 else 'SAI ✗'}")


# Câu 50:
Viết thuật toán tính số điều kiện của $A$. Áp dụng cho ma trận vuông cấp 7.

## Số điều kiện $\kappa_2(A)$

### Bài toán

Cho ma trận vuông khả nghịch $A$ cấp $n$. Tính **số điều kiện** theo chuẩn-2:

$$\kappa_2(A) = \|A\|_2 \, \|A^{-1}\|_2 = \frac{\sigma_{\max}(A)}{\sigma_{\min}(A)}$$

$\kappa_2$ đo mức độ **nhạy cảm** của nghiệm hệ $Ax = b$ với nhiễu: $\kappa$ lớn ⟹ ma trận **gần suy biến**, nghiệm dễ sai lệch.

### Cơ sở lý thuyết

Cả hai giá trị kỳ dị biên đều lấy từ phổ của $G = A^T A$:

$$\sigma_{\max} = \sqrt{\lambda_{\max}(G)}, \qquad \sigma_{\min} = \sqrt{\lambda_{\min}(G)}$$

- $\lambda_{\max}(G)$: **lũy thừa** trực tiếp trên $G$.
- $\lambda_{\min}(G)$: dùng **dịch phổ** — ma trận $G' = \lambda_{\max} I - G$ có giá trị riêng trội $\lambda_{\max} - \lambda_{\min}$. Lũy thừa trên $G'$ cho đại lượng này, suy ra:

$$\lambda_{\min} = \lambda_{\max} - \lambda_{\max}\!\big(\,\lambda_{\max} I - G\,\big)$$

*(Cách khác: lũy thừa nghịch đảo — lũy thừa trên $G^{-1}$ cho $1/\lambda_{\min}$; ở đây dùng dịch phổ để tránh nghịch đảo ma trận.)*

---

### Thuật toán

**Đầu vào:** Ma trận vuông $A$ cấp $n$, sai số $\varepsilon$

**Đầu ra:** $\kappa_2(A)$

**Bước 1.** $G \leftarrow A^T A$.

**Bước 2.** $\lambda_{\max} \leftarrow$ **PowerMethod**$(G)$.

**Bước 3.** $\mu \leftarrow$ **PowerMethod**$(\lambda_{\max} I - G)$; &nbsp; $\lambda_{\min} \leftarrow \lambda_{\max} - \mu$.

**Bước 4.** $\sigma_{\max} = \sqrt{\lambda_{\max}}$, $\;\sigma_{\min} = \sqrt{\lambda_{\min}}$, $\;\kappa_2 = \sigma_{\max}/\sigma_{\min}$.

---

### Lưu ý

- **Vì sao dịch phổ chứ không lũy thừa nghịch đảo:** Lũy thừa nghịch đảo cần giải hệ với $G$ mỗi vòng (hoặc nghịch đảo $G$), tốn kém và kém ổn định khi $G$ gần suy biến. Dịch phổ $\lambda_{\max} I - G$ chỉ là phép trừ ma trận, đơn giản và ổn định.
- **Bình phương số điều kiện:** Vì $\kappa(G) = \kappa(A)^2$, việc tính $\lambda_{\min}(G)$ nhạy sai số gấp đôi so với tính trực tiếp $\sigma_{\min}(A)$. Với ma trận điều kiện xấu ($\kappa$ rất lớn), nên dùng SVD chuẩn.
- **Ý nghĩa ngưỡng:** $\kappa \approx 1$: rất tốt (ma trận trực giao có $\kappa = 1$). $\kappa \gtrsim 10^{k}$: mất khoảng $k$ chữ số chính xác khi giải hệ.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def power_method(A, eps=1e-12, max_iter=5000):
    A = np.array(A, dtype=float)
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        y = A @ x
        lam = x @ y / (x @ x)
        x = y / np.linalg.norm(y)
        if abs(lam - lam_old) < eps:
            return lam, x, k
        lam_old = lam
    return lam, x, max_iter


# ── Số điều kiện κ₂(A) = σmax / σmin ─────────────────────────────
def cond_number(A, eps=1e-12):
    """
    Trả về (κ₂, σmax, σmin).
    σmax = √λmax(AᵀA);  σmin = √λmin(AᵀA) qua dịch phổ.
    """
    A = np.array(A, dtype=float)
    n = len(A)
    G = A.T @ A

    lam_max, _, _ = power_method(G, eps=eps)               # λmax
    # dịch phổ: G' = λmax·I - G  có giá trị riêng trội (λmax - λmin)
    mu, _, _ = power_method(lam_max * np.eye(n) - G, eps=eps)
    lam_min = lam_max - mu                                  # λmin

    smax = np.sqrt(max(lam_max, 0.0))
    smin = np.sqrt(max(lam_min, 0.0))
    kappa = smax / smin if smin > 1e-15 else np.inf
    return kappa, smax, smin


# ====== Áp dụng với ma trận vuông cấp 7 ======
if __name__ == "__main__":
    A = np.array([
        [ 7., 2., 1., 3., 1., 2., 1.],
        [ 2., 8., 2., 1., 2., 1., 3.],
        [ 1., 2., 9., 2., 1., 3., 1.],
        [ 3., 1., 2.,10., 2., 1., 2.],
        [ 1., 2., 1., 2., 6., 2., 1.],
        [ 2., 1., 3., 1., 2.,11., 2.],
        [ 1., 3., 1., 2., 1., 2., 8.],
    ])

    kappa, smax, smin = cond_number(A)
    print(f"σmax = {smax:.8f}")
    print(f"σmin = {smin:.8f}")
    print(f"\nSố điều kiện  κ₂(A) = σmax/σmin = {kappa:.8f}")

    # ── Kiểm tra ──
    kap_np = np.linalg.cond(A, 2)
    print("\n=== Kiểm tra ===")
    print(f"  κ₂ (numpy.cond, p=2) = {kap_np:.8f}")
    print(f"  Sai số so với numpy  = {_sci(abs(kappa - kap_np))}"
          f"  →  {'ĐÚNG ✓' if abs(kappa - kap_np) < 1e-4 else 'SAI ✗'}")
    sv = np.linalg.svd(A, compute_uv=False)
    print(f"  σ (numpy.svd): σmax = {sv[0]:.8f}, σmin = {sv[-1]:.8f}")


# Câu 51:
Viết thuật toán tìm giá trị kỳ dị lớn nhất và vector kỳ dị trái, phải tương ứng của ma trận $A$ bất kỳ.

Nêu ý tưởng của phương pháp xuống thang tìm giá trị riêng trội tiếp theo.

Xấp xỉ ma trận $A = C + aI$ bằng một ma trận $B$ sao cho sai số tương đối không vượt quá 5%, tức là $\dfrac{\lVert A - B \rVert_F}{\lVert A \rVert_F} \le 5\%$, với $a$ là số thứ tự theo danh sách thi, $C$ lấy trong câu 33.

Xây dựng thuật toán sơ lược tìm khai triển kỳ dị của ma trận và sử dụng nó trong bài toán xấp xỉ ảnh.

## Giá trị kỳ dị lớn nhất, xuống thang & xấp xỉ hạng thấp

Câu này gồm nhiều phần. Ma trận dùng: $A = C + aI$ với **$a = 20$** (số thứ tự theo danh sách thi) và $C$ là ma trận cấp 10 ở **Câu 33**.

---

### Phần 1 — σ lớn nhất và vector kỳ dị trái/phải (ma trận bất kỳ)

Giống Câu 48: $\sigma_1 = \sqrt{\lambda_{\max}(A^TA)}$ qua lũy thừa. Vector kỳ dị **phải** $v_1$ là vector riêng của $A^TA$; vector kỳ dị **trái**:

$$u_1 = \frac{A v_1}{\sigma_1}, \qquad A v_1 = \sigma_1 u_1, \quad A^T u_1 = \sigma_1 v_1$$

Với ma trận **không vuông** ($m \neq n$): chọn ma trận Gram nhỏ hơn — $A^TA$ (cấp $n$) nếu $m \geq n$, hoặc $AA^T$ (cấp $m$) nếu $m < n$ — để tiết kiệm.

---

### Phần 2 — Ý tưởng phương pháp xuống thang tìm giá trị riêng trội tiếp theo

Sau khi có cặp trội $(\lambda_1, v_1)$ **chuẩn hoá** của ma trận đối xứng $G$, lập:

$$G_2 = G - \lambda_1 \, v_1 v_1^T \quad\text{(xuống thang Hotelling)}$$

$G_2$ giữ nguyên mọi cặp riêng khác nhưng đưa $\lambda_1 \to 0$, nên giá trị riêng trội của $G_2$ chính là $\lambda_2$. Lặp lại cho $\lambda_3, \lambda_4, \ldots$ — đây chính là cơ chế dùng để bóc lần lượt các giá trị kỳ dị $\sigma_i = \sqrt{\lambda_i}$ (xem Câu 46, 47).

---

### Phần 3 — Xấp xỉ $A \approx B$ với sai số $\dfrac{\|A-B\|_F}{\|A\|_F} \le 5\%$

**Định lý Eckart–Young:** ma trận hạng $\le k$ gần $A$ nhất (theo chuẩn Frobenius) là **SVD cắt cụt** $A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$, với sai số tương đối:

$$\frac{\|A - A_k\|_F}{\|A\|_F} = \sqrt{\frac{\sum_{i>k} \sigma_i^2}{\sum_{i} \sigma_i^2}}$$

Chọn $k$ **nhỏ nhất** sao cho vế phải $\le 5\%$.

> **⚠ Nhận xét quan trọng cho $A = C + 20I$:** vì $a = 20$ **rất lớn** so với các phần tử của $C$ ($\sim 0.05$), ma trận bị chi phối bởi $20I$ nên **mọi** giá trị kỳ dị $\sigma_i \approx 20$ (phổ "phẳng"). Khi đó xấp xỉ hạng thấp **không nén được** — phải giữ gần như đủ hạng mới đạt 5%. Đây là kết luận đúng về bản chất: nén hạng thấp chỉ hiệu quả khi phổ $\sigma$ **suy giảm nhanh**.

---

### Phần 4 — SVD trong xấp xỉ ảnh

Ảnh (ma trận cường độ điểm ảnh) thường có phổ $\sigma$ suy giảm nhanh: một vài giá trị kỳ dị đầu nắm giữ hầu hết "năng lượng". Giữ $k$ giá trị kỳ dị lớn nhất cho **ảnh xấp xỉ**:

$$\text{Ảnh}_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

**Lưu trữ:** thay vì $m \times n$ số, chỉ cần $k(m + n + 1)$ số — nén mạnh khi $k \ll \min(m,n)$. Đây là chỗ SVD phát huy tác dụng thực sự, khác hẳn ma trận phổ phẳng ở Phần 3.

---

### Lưu ý

- **Vì sao Phần 3 và Phần 4 cho kết quả trái ngược:** cùng một thuật toán SVD cắt cụt, nhưng hiệu quả nén **phụ thuộc phổ**. $A = C + aI$ có phổ phẳng ⟹ không nén được; ảnh có phổ dốc ⟹ nén tốt. Bài toán này minh hoạ đúng khi nào nên dùng xấp xỉ hạng thấp.
- **Sai số Eckart–Young theo Frobenius** dùng $\sqrt{\sum_{i>k}\sigma_i^2}$; nếu đo theo chuẩn-2 thì sai số đúng bằng $\sigma_{k+1}$.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def power_method(A, eps=1e-12, max_iter=5000):
    A = np.array(A, dtype=float)
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        y = A @ x
        lam = x @ y / (x @ x)
        x = y / np.linalg.norm(y)
        if abs(lam - lam_old) < eps:
            return lam, x, k
        lam_old = lam
    return lam, x, max_iter


def largest_singular(A, eps=1e-12):
    """σ₁ lớn nhất + vector kỳ dị trái u₁, phải v₁ (ma trận bất kỳ)."""
    A = np.array(A, dtype=float)
    # chọn ma trận Gram nhỏ hơn
    if A.shape[0] >= A.shape[1]:
        G = A.T @ A
        lam, v, k = power_method(G, eps=eps)
        sigma = np.sqrt(max(lam, 0.0))
        u = A @ v / sigma
    else:
        G = A @ A.T
        lam, u, k = power_method(G, eps=eps)
        sigma = np.sqrt(max(lam, 0.0))
        v = A.T @ u / sigma
    return sigma, u, v, k


def svd_deflation(A, rank=None, eps=1e-12):
    """SVD gọn A = U Σ Vᵀ bằng lũy thừa + xuống thang."""
    A = np.array(A, dtype=float)
    m, n = A.shape
    r = min(m, n) if rank is None else rank
    small_left = m < n
    G = A @ A.T if small_left else A.T @ A
    sig, U, V = [], [], []
    for _ in range(r):
        lam, w, _ = power_method(G, eps=eps)
        sigma = np.sqrt(max(lam, 0.0))
        if sigma < 1e-12:
            break
        if small_left:                    # w là vector kỳ dị trái
            u = w; v = A.T @ u / sigma
        else:                             # w là vector kỳ dị phải
            v = w; u = A @ v / sigma
        sig.append(sigma); U.append(u); V.append(v)
        G = G - lam * np.outer(w, w)
    return np.array(sig), np.array(U).T, np.array(V).T


def low_rank_approx(A, tol=0.05, eps=1e-12):
    """
    Xấp xỉ A bằng B hạng thấp sao cho ‖A-B‖_F/‖A‖_F ≤ tol.
    Dùng Eckart–Young: giữ k giá trị kỳ dị lớn nhất.
    """
    A = np.array(A, dtype=float)
    sig, U, V = svd_deflation(A, eps=eps)
    total = np.sum(sig**2)
    # sai số tương đối khi giữ k đầu:  sqrt( Σ_{i>k} σ² / Σ σ² )
    k = 1
    while k < len(sig):
        tail = np.sum(sig[k:]**2)
        if np.sqrt(tail / total) <= tol:
            break
        k += 1
    B = U[:, :k] @ np.diag(sig[:k]) @ V[:, :k].T
    rel = np.linalg.norm(A - B, 'fro') / np.linalg.norm(A, 'fro')
    return B, k, rel, sig


# ── Ma trận C câu 33 (cấp 10) ────────────────────────────────────
C = np.array([
    [0.05953,0.03186,0.09466,0.09514,0.03478,0.047,0.01516,0.09841,0.01701,0.03281],
    [0.06035,0.07194,0.09583,0.08516,0.00064,0.05308,0.08699,0.06794,0.04468,0.06082],
    [0.07307,0.08444,0.06385,0.07472,0.04107,0.08514,0.07277,0.09001,0.07051,0.03304],
    [0.07984,0.04551,0.05665,0.04349,0.05256,0.01644,0.0891,0.01881,0.0014,0.06476],
    [0.05603,0.0098,0.07562,0.04857,0.01058,0.03132,0.06842,0.04374,0.00275,0.09992],
    [0.05876,0.05356,0.01616,0.01209,0.00999,0.07356,0.02366,0.05048,0.08056,0.0216],
    [0.07854,0.0897,0.03872,0.03786,0.03373,0.00599,0.08687,0.04372,0.05682,0.0629],
    [0.07909,0.03647,0.09861,0.04025,0.03746,0.00255,0.04359,0.06612,0.04205,0.02439],
    [0.02533,0.09595,0.00065,0.04671,0.03464,0.04578,0.07115,0.04019,0.04568,0.01673],
    [0.00902,0.09266,0.09269,0.07965,0.05633,0.00469,0.00403,0.07378,0.04205,0.06873],
], dtype=float)

a = 20            # số thứ tự theo danh sách thi
A = C + a * np.eye(len(C))


if __name__ == "__main__":
    print("="*64)
    print("PHẦN 1 — Giá trị kỳ dị lớn nhất & vector kỳ dị (A = C + 20I)")
    print("="*64)
    sigma, u, v, k = largest_singular(A)
    print(f"σ₁ = {sigma:.8f}   ({k} vòng)")
    print(f"‖A v₁ - σ₁ u₁‖  = {_sci(np.linalg.norm(A@v - sigma*u))}")
    print(f"‖Aᵀ u₁ - σ₁ v₁‖ = {_sci(np.linalg.norm(A.T@u - sigma*v))}")
    print(f"σ₁ (numpy)      = {np.linalg.svd(A, compute_uv=False)[0]:.8f}")

    print("\n" + "="*64)
    print("PHẦN 2 — Xấp xỉ A ≈ B, hạng thấp, ‖A-B‖_F/‖A‖_F ≤ 5%")
    print("="*64)
    B, kk, rel, sig = low_rank_approx(A, tol=0.05)
    print("Các giá trị kỳ dị:", "  ".join(f"{s:.4f}" for s in sig))
    print(f"\nHạng nhỏ nhất thoả 5% : k = {kk}")
    print(f"Sai số tương đối       : {rel*100:.4f}%   →  {'ĐẠT ✓' if rel<=0.05 else 'CHƯA'}")

    # đối chiếu sai số lý thuyết Eckart–Young
    total = np.sum(sig**2)
    for kt in range(1, len(sig)+1):
        e = np.sqrt(np.sum(sig[kt:]**2)/total)
        mark = "  ← chọn" if kt == kk else ""
        print(f"   k={kt}:  sai số = {e*100:6.3f}%{mark}")

    print("\n  ⚠ Nhận xét: A = C + 20I bị chi phối bởi 20I nên các σ_i ≈ 20 "
          "(phổ 'phẳng').\n    Xấp xỉ hạng thấp KHÔNG nén được — phải giữ gần đủ hạng. "
          "Nén\n    hạng thấp chỉ hiệu quả khi phổ σ suy giảm nhanh (như ảnh dưới đây).")

    print("\n" + "="*64)
    print("PHẦN 3 — SVD trong xấp xỉ (nén) ẢNH")
    print("="*64)
    # Ảnh mô phỏng: cấu trúc hạng thấp + chi tiết (phổ σ suy giảm nhanh)
    n = 64
    t = np.linspace(0, 3*np.pi, n)
    img = np.outer(np.sin(t), np.cos(t)) + 0.6*np.outer(t, np.ones(n)) \
        + 0.3*np.outer(np.ones(n), np.sin(2*t))
    img += 0.05 * np.random.default_rng(0).standard_normal((n, n))

    sig_img, U_img, V_img = svd_deflation(img, rank=n)
    tot = np.sum(sig_img**2)
    print(f"Ảnh {n}×{n}: {len(sig_img)} giá trị kỳ dị, σ₁={sig_img[0]:.2f}, "
          f"σ_cuối={sig_img[-1]:.4f}")
    print(f"{'hạng k':>7}  {'sai số':>9}  {'tỉ lệ lưu trữ':>14}")
    for k_img in [1, 2, 3, 5, 10, 20]:
        Ik = U_img[:, :k_img] @ np.diag(sig_img[:k_img]) @ V_img[:, :k_img].T
        rel_img = np.linalg.norm(img - Ik, 'fro') / np.linalg.norm(img, 'fro')
        store = k_img * (2*n + 1) / (n*n)          # (U,Σ,V) so với ảnh gốc
        print(f"{k_img:>7}  {rel_img*100:8.3f}%  {store*100:12.1f}%")


# Câu 52:
Viết thuật toán tìm giá trị kỳ dị lớn nhất và vector kỳ dị trái, phải của ma trận $A$ bất kỳ. Nêu ý tưởng của phương pháp xuống thang tìm giá trị riêng trội tiếp theo. Xác định ba giá trị kỳ dị lớn nhất của ma trận $A = C + aI$ và xấp xỉ $A$ qua ba giá trị kỳ dị đó ($a$ là số thứ tự của bạn theo danh sách thi, $C$ cho ở câu 34).

## σ lớn nhất, xuống thang & xấp xỉ hạng-3

Ma trận: $A = C + aI$ với **$a = 20$** và $C$ là ma trận cấp 7 ở **Câu 34**.

---

### Phần 1 — σ lớn nhất và vector kỳ dị trái/phải

Giống Câu 48/51: $\sigma_1 = \sqrt{\lambda_{\max}(A^TA)}$ bằng lũy thừa; $v_1$ là vector kỳ dị phải, $u_1 = A v_1 / \sigma_1$ là vector kỳ dị trái. Thoả $A v_1 = \sigma_1 u_1$ và $A^T u_1 = \sigma_1 v_1$.

---

### Phần 2 — Ý tưởng phương pháp xuống thang tìm giá trị riêng trội tiếp theo

Với ma trận đối xứng $G = A^TA$, sau khi có $(\lambda_1, v_1)$ chuẩn hoá, lập:

$$G_2 = G - \lambda_1 \, v_1 v_1^T$$

$G_2$ đưa $\lambda_1 \to 0$ nhưng giữ nguyên các cặp riêng khác ⟹ trội của $G_2$ là $\lambda_2$. Lặp lại cho $\lambda_3$. Mỗi $\sigma_i = \sqrt{\lambda_i}$.

---

### Phần 3 — Ba giá trị kỳ dị lớn nhất & xấp xỉ hạng-3

Áp dụng **lũy thừa + xuống thang** trên $A^TA$ ba lần để lấy $\sigma_1, \sigma_2, \sigma_3$ (và $u_i, v_i$). Xấp xỉ hạng-3 (SVD cắt cụt — tối ưu theo Eckart–Young):

$$A_3 = \sum_{i=1}^{3} \sigma_i \, u_i v_i^T$$

Sai số tương đối:

$$\frac{\|A - A_3\|_F}{\|A\|_F} = \sqrt{\frac{\sigma_4^2 + \cdots + \sigma_7^2}{\sigma_1^2 + \cdots + \sigma_7^2}}$$

---

### Lưu ý

- **Kết quả với $A = C + 20I$:** vì $a = 20$ lớn, cả 7 giá trị kỳ dị đều $\approx 20$ (phổ gần **phẳng**). Xấp xỉ hạng-3 chỉ nắm khoảng $1/4$ "năng lượng", nên sai số $\approx 75\%$ — **lớn hơn nhiều** so với ngưỡng 5%. Đây là kết luận đúng: **hạng-3 không đủ** cho ma trận có phổ phẳng.

- **So sánh với xấp xỉ ảnh (Câu 51):** cùng thuật toán, nhưng ảnh có phổ dốc nên hạng-3 đã nén rất tốt. Điều làm nên hiệu quả nén là **hình dạng phổ $\sigma$**, không phải bản thân thuật toán.

- **Kiểm chứng:** ba $\sigma$ tính được khớp `numpy.linalg.svd` tới $\sim 10^{-12}$, và sai số hạng-3 khớp chính xác công thức Eckart–Young.

In [ ]:
import numpy as np


# ── Định dạng số khoa học kiểu 1.23×10⁻⁵ ─────────────────────────
def _sci(x, sig=4):
    if abs(x) < 1e-14:
        return "0"
    s = f"{x:.{sig}e}"
    m, e = s.split("e")
    exp = int(e)
    sup = str(abs(exp)).translate(str.maketrans("0123456789", "⁰¹²³⁴⁵⁶⁷⁸⁹"))
    sign = "⁻" if exp < 0 else ""
    return f"{m}×10{sign}{sup}"


def power_method(A, eps=1e-12, max_iter=5000):
    A = np.array(A, dtype=float)
    n = len(A)
    x = np.ones(n) / np.sqrt(n)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        y = A @ x
        lam = x @ y / (x @ x)
        x = y / np.linalg.norm(y)
        if abs(lam - lam_old) < eps:
            return lam, x, k
        lam_old = lam
    return lam, x, max_iter


def largest_singular(A, eps=1e-12):
    """σ₁ + vector kỳ dị trái/phải (ma trận bất kỳ)."""
    A = np.array(A, dtype=float)
    if A.shape[0] >= A.shape[1]:
        lam, v, k = power_method(A.T @ A, eps=eps)
        sigma = np.sqrt(max(lam, 0.0)); u = A @ v / sigma
    else:
        lam, u, k = power_method(A @ A.T, eps=eps)
        sigma = np.sqrt(max(lam, 0.0)); v = A.T @ u / sigma
    return sigma, u, v, k


def svd_deflation(A, rank=None, eps=1e-12):
    """SVD gọn bằng lũy thừa + xuống thang; lấy 'rank' giá trị kỳ dị đầu."""
    A = np.array(A, dtype=float)
    m, n = A.shape
    r = min(m, n) if rank is None else rank
    small_left = m < n
    G = A @ A.T if small_left else A.T @ A
    sig, U, V = [], [], []
    for _ in range(r):
        lam, w, _ = power_method(G, eps=eps)
        sigma = np.sqrt(max(lam, 0.0))
        if sigma < 1e-12:
            break
        if small_left:
            u = w; v = A.T @ u / sigma
        else:
            v = w; u = A @ v / sigma
        sig.append(sigma); U.append(u); V.append(v)
        G = G - lam * np.outer(w, w)
    return np.array(sig), np.array(U).T, np.array(V).T


# ── Ma trận C câu 34 (cấp 7) ─────────────────────────────────────
C = np.array([
    [.1588, .0064, .0025, .0304, .0014, .0083, .1594],
    [.0057, .2645, .0436, .0099, .0083, .0201, .3413],
    [.0264, .1506, .3557, .0139, .0142, .0070, .0236],
    [.3299, .0565, .0495, .3636, .0204, .0483, .0649],
    [.0089, .0081, .0333, .0295, .3412, .0237, .0020],
    [.1190, .0901, .0996, .1260, .1722, .2368, .3369],
    [.0063, .0126, .0196, .0098, .0064, .0132, .0012],
], dtype=float)

a = 20
A = C + a * np.eye(len(C))


if __name__ == "__main__":
    print("="*64)
    print("PHẦN 1 — σ lớn nhất & vector kỳ dị (A = C + 20I, cấp 7)")
    print("="*64)
    sigma, u, v, k = largest_singular(A)
    print(f"σ₁ = {sigma:.8f}   ({k} vòng)")
    print(f"‖A v₁ - σ₁ u₁‖  = {_sci(np.linalg.norm(A@v - sigma*u))}")
    print(f"‖Aᵀ u₁ - σ₁ v₁‖ = {_sci(np.linalg.norm(A.T@u - sigma*v))}")

    print("\n" + "="*64)
    print("PHẦN 2 — BA giá trị kỳ dị lớn nhất & xấp xỉ hạng-3")
    print("="*64)
    sig3, U3, V3 = svd_deflation(A, rank=3)
    print("3 giá trị kỳ dị lớn nhất:", "  ".join(f"σ{i+1} = {s:.6f}" for i, s in enumerate(sig3)))

    A3 = U3 @ np.diag(sig3) @ V3.T                 # xấp xỉ hạng 3
    rel = np.linalg.norm(A - A3, 'fro') / np.linalg.norm(A, 'fro')
    print(f"\nXấp xỉ hạng-3 A₃ = Σ_{{i=1..3}} σ_i u_i v_iᵀ")
    print(f"Sai số tương đối ‖A - A₃‖_F/‖A‖_F = {rel*100:.4f}%")

    # ── Kiểm tra so với numpy ──
    sig_np = np.linalg.svd(A, compute_uv=False)
    print("\n=== Kiểm tra ===")
    print(f"  σ (numpy, đủ 7): " + "  ".join(f"{s:.4f}" for s in sig_np))
    print(f"  Sai số 3 σ đầu so numpy = {_sci(np.max(np.abs(sig3 - sig_np[:3])))}"
          f"  →  {'ĐÚNG ✓' if np.max(np.abs(sig3-sig_np[:3]))<1e-6 else 'SAI ✗'}")
    # sai số lý thuyết Eckart–Young cho hạng 3
    ey = np.sqrt(np.sum(sig_np[3:]**2) / np.sum(sig_np**2))
    print(f"  Sai số Eckart–Young (lý thuyết) = {ey*100:.4f}%   "
          f"(khớp: {'✓' if abs(ey-rel)<1e-6 else '✗'})")

    print(f"\n  ⚠ Vì A = C + 20I (phổ σ ≈ 20, gần phẳng), hạng-3 chỉ "
          f"nắm ~{(1-rel)*100:.1f}% năng lượng;\n    sai số {rel*100:.1f}% "
          f"lớn hơn 5% — cần nhiều hạng hơn để xấp xỉ tốt.")
